# 8.11 裁判會因答案排列而改判嗎？


把兩份作業放桌上，評分者不論內容都選左邊，那麼只要交換位置，勝者就變了。比較模型回答時，畫面上的A、B也可能產生這種偏差。要測它，不能只看兩次都選A就說「判斷穩定」，因為第二次的A可能已是另一個回答。

前置是[8.10以已知例子核對裁判](https://birdhackor.github.io/tiny-perceptron-vlm/8.10.html)與[W.2清單索引](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。本節要區分回答的身分與顯示位置。回答0是「2+2=4」，回答1是「2+2=5」。位置A只是這次先呈現的選項，交換之後不再代表同一份內容。判斷是否一致，應把選擇轉回原回答的身分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
answers = {"correct": "2+2=4", "wrong": "2+2=5"}
original_order = ["correct", "wrong"]
swapped_order = original_order[::-1]


def choose_first(order):
    return order[0]


first = choose_first(original_order)
second = choose_first(swapped_order)
print("原順序選", first, answers[first])
print("交換後選", second, answers[second])
print("內容身分一致", first == second)

輸入字典給兩份回答不會隨排序改變的名稱；兩個清單則表示畫面順序。`[::-1]`把清單反轉，所以原順序是correct、wrong，交換順序是wrong、correct。函式`choose_first`故意寫成永遠選第一個，模擬一個有位置偏差的裁判，並沒有讀內容判斷算術。

第一行輸出`原順序選 correct 2+2=4`，第二行是`交換後選 wrong 2+2=5`，第三行`False`。兩次都選畫面A，但得到不同內容。若只記字母A而沒有記那次A對應哪個回答，你會把這個失敗誤看成一致。實測應保存每次提示與排列，再把結果映射到同一份回答。

這個小實驗沒有呼叫外部評分服務，也沒有證明任何實際裁判一定有偏差；它讓我們知道該如何找到偏差。更完整的測試會對很多對答案交換順序，統計映射後一致的比例。若裁判允許平手，也要事先決定如何記錄，不能事後把平手只算到有利的一邊。

假如第一次畫面A是正確短答、第二次畫面B才是同一正確短答，兩次合理選擇的字母就應不同。這和「回答身分一致」完全不矛盾。收集結果時可以記兩欄：顯示位置以及原回答ID；ID只是辨識這一份內容的名稱，像程式裡的correct與wrong。交換實驗還要保持評分規則和問題原文不變，否則你一次改了多個條件，位置就不再是唯一可追查的原因。

練習把函式內容改成`return "correct"`，先預測原順序與交換後都會選正確回答，第三行為`True`，再執行核對。這個函式同樣是手寫理想規則，並非能理解新答案的裁判。它的作用是對照：好的內容判斷應保持回答身分一致，即使畫面位置改變。往後你用模型評分時，也可以沿用這個交換順序的檢查。
